# Juliet 최종 평가

`.env`의 `OPENROUTER_API_KEY`만 사용합니다. test case 하나의 E1/E3/E4/E5/E6 탐지는 한 batch API 요청으로 처리합니다. 저장된 실제 결과로 Single, Fixed-2, Utility Top-2, Adaptive Top-2→Full-5, Full-5를 비교하며, patch도 case의 승인된 finding들을 한 요청으로 묶습니다.

In [ ]:
from pathlib import Path
EVAL_ROOT = Path.cwd().resolve()
if EVAL_ROOT.name != 'Model_Evaluation':
    EVAL_ROOT = (EVAL_ROOT / 'Model_Evaluation').resolve()
CONFIG_PATH = EVAL_ROOT / 'configs' / 'full.toml'
ENV_FILE = EVAL_ROOT.parent / '.env'
ARTIFACT = EVAL_ROOT / 'artifacts' / 'juliet_utility_router.pkl'
TEST_CASE_LIMIT = 0  # 0 = frozen test split 전체
MAX_CANDIDATES_PER_CASE = 4
HARD_NEGATIVES_PER_CASE = 1
PATCH_CASE_LIMIT = 0  # 0 = 탐지된 test case 전체


In [ ]:
import json, shutil, sys
sys.path.insert(0, str(EVAL_ROOT / 'src'))
from model_evaluation.config import load_config, load_mapping
from model_evaluation.stages.materialize_dataset import materialize_dataset
from model_evaluation.candidates import cache_candidates
from model_evaluation.workflow import plan_outcome_matrix, collect_outcome_matrix, audit_outcome_matrix, evaluate_utility_router
from model_evaluation.live_evaluation import run_batched_patch_evaluation
from model_evaluation.adapters.llm_security import activate_parent_package
activate_parent_package()
from llm_security.routing import BudgetedUtilityRouter
config = load_config(CONFIG_PATH)
mapping = load_mapping(config.paths.mapping)
RUN_DIR = EVAL_ROOT / 'work' / 'router_evaluation'
RESULT_DIR = EVAL_ROOT / 'results' / 'router_evaluation'
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
if not ARTIFACT.exists():
    raise FileNotFoundError(f'train.ipynb를 먼저 끝까지 실행하세요: {ARTIFACT}')
router = BudgetedUtilityRouter.load(ARTIFACT)
models = sorted({item.model_id for item in router.assignments.values()})
if len(models) != 1:
    raise ValueError('artifact는 case당 한 번 호출하는 단일 physical model이어야 합니다')
print('Physical model:', models[0])

## 1. Frozen test case와 candidate cache

In [ ]:
materialization = materialize_dataset(
    config, mapping, output_directory=RUN_DIR / 'cases', splits=('test',),
    limits={'test': TEST_CASE_LIMIT}, progress=print,
)
candidate_summary = cache_candidates(
    RUN_DIR / 'cases' / 'cases_test.jsonl', RUN_DIR / 'candidates' / 'candidates_test.jsonl',
    max_source_bytes=config.max_source_bytes, parse_timeout_ms=config.parse_timeout_ms, progress=print,
)
print(json.dumps({'materialization': materialization, 'candidates': candidate_summary}, ensure_ascii=False, indent=2))

## 2. Test 호출 계획 — case당 탐지 API 최대 1회

In [ ]:
plan = plan_outcome_matrix(
    cases_path=RUN_DIR / 'cases' / 'cases_test.jsonl',
    candidate_cache=RUN_DIR / 'candidates' / 'candidates_test.jsonl',
    selection_manifest=RUN_DIR / 'selections' / 'selected_test.jsonl',
    outcome_path=RUN_DIR / 'outcomes' / 'outcomes_test.jsonl', model_ids=models,
    max_candidates_per_case=MAX_CANDIDATES_PER_CASE, hard_negatives_per_case=HARD_NEGATIVES_PER_CASE,
)
print(json.dumps(plan, ensure_ascii=False, indent=2))

## 3. 실제 test batch 탐지 수집

In [ ]:
collection = collect_outcome_matrix(
    env_file=ENV_FILE, cases_path=RUN_DIR / 'cases' / 'cases_test.jsonl',
    candidate_cache=RUN_DIR / 'candidates' / 'candidates_test.jsonl',
    outcome_path=RUN_DIR / 'outcomes' / 'outcomes_test.jsonl',
    ledger_path=RUN_DIR / 'ledgers' / 'test_api_ledger.jsonl', model_ids=models,
    max_candidates_per_case=MAX_CANDIDATES_PER_CASE, hard_negatives_per_case=HARD_NEGATIVES_PER_CASE,
)
print(json.dumps(collection, ensure_ascii=False, indent=2))

## 4. 정책 비교와 전체 End-to-End 평가 — 추가 API 없음

In [ ]:
test_outcomes = RUN_DIR / 'outcomes' / 'outcomes_test.jsonl'
matrix_audit = audit_outcome_matrix(
    test_outcomes, expected_assignment_ids=list(router.assignments),
    selection_manifest=RUN_DIR / 'selections' / 'selected_test.jsonl',
)
print(json.dumps(matrix_audit, ensure_ascii=False, indent=2))
if not matrix_audit['complete']:
    raise RuntimeError('test 수집이 중단되었습니다. 3번 셀을 다시 실행하면 이어서 진행합니다.')
final_report = evaluate_utility_router(
    artifact_path=ARTIFACT, test_outcomes=test_outcomes,
    test_cases=RUN_DIR / 'cases' / 'cases_test.jsonl',
    candidate_cache=RUN_DIR / 'candidates' / 'candidates_test.jsonl',
    selection_manifest=RUN_DIR / 'selections' / 'selected_test.jsonl',
    report_path=RESULT_DIR / 'policy_and_end_to_end.json',
    candidate_gate_enabled=False, max_candidates_per_case=MAX_CANDIDATES_PER_CASE,
)
print(json.dumps(final_report, ensure_ascii=False, indent=2))

## 5. Case 단위 batch patch 생성 및 검증

한 case의 승인된 finding을 하나의 patch 요청으로 묶습니다. `make`가 있으면 Juliet build를 실행하고, 현재처럼 compiler가 없으면 격리 복사본에 patch가 정상 적용되는지만 검증합니다.

In [ ]:
if shutil.which('make'):
    patch_commands = [{'name': 'juliet-build', 'command': ['make'], 'timeout_seconds': 300}]
    verification_level = 'compile/test'
else:
    patch_commands = []
    verification_level = 'apply-and-diff-check-only'
patch_report = run_batched_patch_evaluation(
    env_file=ENV_FILE,
    detection_path=RUN_DIR / 'outcomes' / 'outcomes_test.detections.jsonl',
    output_path=RUN_DIR / 'patches' / 'patch_results.jsonl',
    ledger_path=RUN_DIR / 'ledgers' / 'patch_api_ledger.jsonl',
    commands=patch_commands, max_cases=PATCH_CASE_LIMIT,
)
patch_report['verification_level'] = verification_level
print(json.dumps(patch_report, ensure_ascii=False, indent=2))